# Resume AI Assistant (Structured Output)

Tailors a resume to a target job description — running a **gap analysis**, rewriting the resume, and drafting a cover letter — with results returned as **schema-validated structured output**.

**What it demonstrates**
- Enforcing reliable JSON output with Pydantic models
- Multi-step LLM workflows (analyse → rewrite → generate)
- Running the same pipeline across OpenAI and Gemini

**Stack:** Python · OpenAI · Google Gemini · Pydantic


In [1]:
!pip install pydantic
from pydantic import BaseModel

In [7]:
class User(BaseModel):
    name: str
    age: float
    email: str

In [8]:
user = User(name = "Alice", age = 30, email = "alice@example.com")
print(user.model_dump_json())

{"name":"Alice","age":30.0,"email":"alice@example.com"}


In [33]:
import os
from openai import OpenAI
from google import genai
from dotenv import load_dotenv
import json

from typing import Optional, List, Union, Dict
from IPython.display import Markdown, display



load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

openai_client = OpenAI(api_key = openai_api_key)
googleai_client = genai.Client(api_key=google_api_key)

# Initialize the Gemini model, choose a suitable model like "gemini-2.0-flash"
gemini_model = "gemini-2.5-flash"


In [ ]:
class Scientist(BaseModel):
    name: str
    field: str
    known_for: list[str]
    birth_year: int

In [43]:
class Destination(BaseModel):
    city: str
    country: str
    top_attractions: list[str]


In [25]:
prompt = """
Give me a json object with details about a famous scientist.
Include the following fields: name, field, known_for (a list of their major contributions), and birth_year.
"""

response = openai_client.chat.completions.parse(model="gpt-4o",
messages = [{"role": "user", "content": prompt}],
temperature = 0,
response_format = Scientist,
max_tokens = 500)

In [50]:
prompt = """
Give me a json object with details about a popular travel destination in Africa.
Include the following fields: city, country, and top_attractions.
"""

response = openai_client.chat.completions.parse(model="gpt-4o",
messages = [{"role": "user", "content": prompt}],
temperature = 0,
response_format = Destination,
max_tokens = 500)

In [51]:
response.choices[0].message.content

'{"city":"Cape Town","country":"South Africa","top_attractions":["Table Mountain","Robben Island","Kirstenbosch National Botanical Garden","Cape of Good Hope","Boulders Beach","Victoria & Alfred Waterfront","District Six Museum","Chapman\'s Peak Drive","Two Oceans Aquarium","Bo-Kaap"]}'

In [52]:
json.loads(response.choices[0].message.content)

{'city': 'Cape Town',
 'country': 'South Africa',
 'top_attractions': ['Table Mountain',
  'Robben Island',
  'Kirstenbosch National Botanical Garden',
  'Cape of Good Hope',
  'Boulders Beach',
  'Victoria & Alfred Waterfront',
  'District Six Museum',
  "Chapman's Peak Drive",
  'Two Oceans Aquarium',
  'Bo-Kaap']}

In [42]:
response = googleai_client.models.generate_content(
    model=gemini_model,
    contents = prompt)

print(response.text)

```json
{
  "name": "Marie Skłodowska-Curie",
  "field": "Physics, Chemistry",
  "known_for": [
    "Pioneering research on radioactivity",
    "Discovery of the elements polonium and radium",
    "First woman to win a Nobel Prize",
    "Only person to win Nobel Prizes in two different scientific fields",
    "First female professor at the University of Paris"
  ],
  "birth_year": 1867
}
```


In [53]:
def print_markdown(text):
    """Displays text as Markdown."""
    display(Markdown(text))

In [54]:
# Let's define a sample resume text
resume_text = """
**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.

**Experience**

**Marketing Asssistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and social media updates to improve audience engagement.
- Managed social media accounts and grew follower numbers.
- Supported coordination of marketing events.
- Conducted market research and competitor analysis.

**Skils**
- Digital Marketing (SEO basics, Email Marketing)
- Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021
"""


In [55]:
# Let's define a sample job description text
job_description_text = """
# Job Title: Digital Marketing Specialist

**Company:** BrightWave Digital Agency

**Location:** Toronto, ON

## About Us:
BrightWave Digital Agency creates digital marketing campaigns for a variety of clients. We are looking for a Digital Marketing Specialist to join our team and assist in managing campaigns.

## Responsibilities:
- Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
- Use Google Analytics to measure performance and prepare basic performance reports.
- Support social media management tasks including content scheduling and community engagement.
- Perform keyword research and assist in optimizing content for SEO.
- Work with designers to help coordinate campaign materials.
- Keep informed about current digital marketing trends.

## Qualifications:
- Bachelor's degree in Marketing, Communications, or similar.
- 2+ years of digital marketing experience.
- Familiarity with SEO, SEM, Google Analytics, and social media.
- Ability to interpret basic marketing data.
- Good communication and writing skills.
- Knowledge of CRM systems (e.g., HubSpot) helpful.
- Experience with Adobe Creative Suite is beneficial.
"""

In [56]:
# Let's display the original resume 
print_markdown("**--- Original Resume ---**")
print_markdown(resume_text)

**--- Original Resume ---**


**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Marketing professsional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.

**Experience**

**Marketing Asssistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Assisted with digital marketing campaigns including email and social media.
- Created blog posts and social media updates to improve audience engagement.
- Managed social media accounts and grew follower numbers.
- Supported coordination of marketing events.
- Conducted market research and competitor analysis.

**Skils**
- Digital Marketing (SEO basics, Email Marketing)
- Social Media Tools (Hootsuite, Buffer)
- Microsoft Office Suite, Google Workspace
- Basic knowledge of Adobe Photoshop

**Education**  
**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021


In [57]:
# Let's display the target job desciption
print_markdown("\n**--- Target Job Description ---**")
print_markdown(job_description_text)


**--- Target Job Description ---**


# Job Title: Digital Marketing Specialist

**Company:** BrightWave Digital Agency

**Location:** Toronto, ON

## About Us:
BrightWave Digital Agency creates digital marketing campaigns for a variety of clients. We are looking for a Digital Marketing Specialist to join our team and assist in managing campaigns.

## Responsibilities:
- Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
- Use Google Analytics to measure performance and prepare basic performance reports.
- Support social media management tasks including content scheduling and community engagement.
- Perform keyword research and assist in optimizing content for SEO.
- Work with designers to help coordinate campaign materials.
- Keep informed about current digital marketing trends.

## Qualifications:
- Bachelor's degree in Marketing, Communications, or similar.
- 2+ years of digital marketing experience.
- Familiarity with SEO, SEM, Google Analytics, and social media.
- Ability to interpret basic marketing data.
- Good communication and writing skills.
- Knowledge of CRM systems (e.g., HubSpot) helpful.
- Experience with Adobe Creative Suite is beneficial.


In [ ]:
def openai_resume_rewrite(prompt: str, model: str = "gpt-4o", temperature: float = 0.7, max_tokens: int = 1500, response_format: Optional[dict] = None) -> str | dict:

    """
    Generate text using OpenAI API

    This function sends a prompt to OpenAI's API and returns the generated response.
    It supports both standard text generation and structured parsing with response_format.

    Args:
        prompt (str): The prompt to send to the model, i.e.: your instructions for the AI
        model (str): The OpenAI model to use (default: "gpt-4o")
        temperature (float): Controls randomness, where lower values make output more deterministic
        max_tokens (int): Maximum number of tokens to generate, which limits the response length
        response_format (dict): Optional format specification
        In simple terms, response_format is optional. If the user gives me a dictionary, cool! 
        If they don't give me anything, just assume it's None and keep going."

    Returns:
        str or dict: The generated text or parsed structured data, depending on response_format
    """

    try:
        if not response_format:
            response = openai_client.chat.completions.create(model=model,
            messages = [
                {"role": "user",
                "content": "You are a helpful assistant specializing in resume writing and career advice."
                },
                {"role": "user", "content": prompt}
                ],
            temperature = temperature,
            max_tokens = max_tokens)
            return response.choices[0].message.content

        else:
            response = openai_client.chat.completions.parse(model=model,
            messages = [
                {"role": "user",
                "content": "You are a helpful assistant specializing in resume writing and career advice."
                },
                {"role": "user", "content": prompt}
                ],
            temperature = temperature,
            response_format = response_format)

            return response.choices[0].message.parsed

    except Exception as e:
        return f"An error occurred: {e}"

In [68]:
from google.genai import types

def gemini_resume_rewrite(prompt: str, model: str = "gemini-2.5-flash", temperature: float = 0.7, max_tokens: int = 3000, response_format: Optional[dict] = None) -> str | dict:

    try:
        config = types.GenerateContentConfig(
            system_instruction="You are a helpful assistant specializing in resume writing and career advice.",
            temperature=temperature,
            max_output_tokens=max_tokens
        )

        if not response_format:
            response = googleai_client.models.generate_content(
                model=model,
                contents=prompt,
                config=config
            )
            return response.text

        else:
            structured_config = types.GenerateContentConfig(
                system_instruction="You are a helpful assistant specializing in resume writing and career advice.",
                temperature=temperature,
                max_output_tokens=max_tokens,
                response_mime_type="application/json",
                response_schema=response_format  # Uses the schema passed in
            )
            response = googleai_client.models.generate_content(
                model=model,
                contents=prompt,
                config=structured_config
            )
            return json.loads(response.text)

    except Exception as e:
        return f"An error occurred: {e}"

In [63]:
prompt = f"""
Context:
You are a professional resume writer helping a candidate tailor their resume for a specific job opportunity. The resume and job description are provided below.

Instruction:
Enhance the resume to make it more impactful. Focus on:
- Highlighting relevant skills and achievements.
- Using strong action verbs and quantifiable results where possible.
- Rewriting vague or generic bullet points to be specific and results-driven.
- Emphasizing experience and skills most relevant to the job description.
- Reorganizing sections if necessary to better match the job requirements.

Resume:
{resume_text}

Output:
Provide a revised and improved version of the resume that is well-formatted. Only return the updated resume.
"""


In [61]:
# Get response from OpenAI API
openai_output = openai_resume_rewrite(prompt, temperature = 0.7)

# Display the results
print_markdown("#### OpenAI Response:")
print_markdown(openai_output)

#### OpenAI Response:

**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

---

**Summary**  
Results-driven marketing professional with over 2 years of experience in executing digital campaigns, content creation, and social media management. Proven ability to enhance online presence and drive audience engagement through innovative strategies and meticulous execution.

---

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**  
- Spearheaded digital marketing efforts, increasing email campaign open rates by 25% through strategic content optimization and A/B testing.
- Developed and curated engaging blog content and social media posts, boosting audience interaction by 40%.
- Expanded social media follower base by 30% over 12 months by implementing targeted growth strategies.
- Played a key role in organizing successful marketing events, enhancing brand visibility and customer engagement.
- Conducted comprehensive market research and competitor analysis, providing actionable insights that informed strategic planning.

---

**Skills**
- Advanced Digital Marketing (SEO techniques, Email Marketing strategies)
- Proficient in Social Media Management Tools (Hootsuite, Buffer)
- Expertise in Microsoft Office Suite and Google Workspace
- Working knowledge of Adobe Photoshop for basic graphic design

---

**Education**  
**Bachelor of Commerce, Marketing**  
Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021  

---

In [64]:
# Get response from Gemini API
gemini_output = gemini_resume_rewrite(prompt, temperature = 0.7)

# Display the results
print_markdown("#### Gemini Response:")
print_markdown(gemini_output)

#### Gemini Response:

**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Dynamic Marketing Professional with 2 years of experience driving digital campaigns, creating engaging content, and managing social media presence for increased brand visibility and audience engagement. Adept at leveraging market insights and digital tools to support comprehensive marketing strategies and achieve measurable results.

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
*   Contributed to the execution of 10+ digital marketing campaigns, including email newsletters and social media advertising, resulting in an average 15% increase in click-through rates (CTR) and expanded customer reach.
*   Developed and published engaging content, including 20+ blog posts and daily social media updates across Instagram, Facebook, and Twitter, increasing overall audience engagement by 20%.
*   Managed and strategically grew social media accounts, increasing total follower count by 25% (e.g., from 4,000 to 5,000) and enhancing online brand presence.
*   Supported the successful coordination and promotion of 5+ marketing events, managing logistics and communication to maximize attendance and brand exposure.
*   Conducted comprehensive market research and competitor analysis, providing actionable insights that informed content strategy and campaign development.

**Skills**
*   **Digital Marketing:** Email Marketing, Social Media Marketing, Content Creation, SEO Fundamentals, Campaign Support, Analytics

In [65]:
# Prompt to analyze the resume against the job description

def analyze_resume_against_job_description(job_description_text: str, resume_text: str, model: str = "openai") -> str:
    """
    Analyze the resume against the job description and return a structured comparison.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        model (str): The model to use for analysis ("openai" or "gemini").

    Returns:
        str: A clear, structured comparison of the resume and job description.
    """
    # This prompt instructs the AI to act as a career advisor and analyze how well the resume matches the job description
    # It asks for a structured analysis with 4 specific sections: requirements, matches, gaps, and strengths
    prompt = f"""
    Context:
    You are a career advisor and resume expert. Your task is to analyze a candidate's resume against a specific job description to assess alignment and identify areas for improvement.

    Instruction:
    Review the provided Job Description and Resume. Identify key skills, experiences, and qualifications in the Job Description and compare them to what's present in the Resume. Provide a structured analysis with the following sections:
    1. **Key Requirements from Job Description:** List the main skills, experiences, and qualifications sought by the employer.
    2. **Relevant Experience in Resume:** List the skills and experiences from the resume that match or align closely with the job requirements.
    3. **Gaps/Mismatches:** Identify important skills or qualifications from the Job Description that are missing, unclear, or underrepresented in the Resume.
    4. **Potential Strengths:** Highlight any valuable skills, experiences, or accomplishments in the resume that are not explicitly requested in the job description but could strengthen the application.

    Job Description:

    {job_description_text}

    Resume:

    {resume_text}

    Output:
    Return a clear, structured comparison with the four sections outlined above.
    """

    # This conditional block selects which AI model to use based on the 'model' parameter
    if model == "openai":
        # Uses OpenAI's model to generate the gap analysis with moderate creativity (temperature=0.7)
        gap_analysis = openai_resume_rewrite(prompt, temperature=0.7)
    elif model == "gemini":
        # Uses Google's Gemini model with less creativity (temperature=0.5) for more focused results
        gap_analysis = gemini_resume_rewrite(prompt, temperature=0.5)
    else:
        # Raises an error if an invalid model name is provided
        raise ValueError(f"Invalid model: {model}")

    # Returns the generated gap analysis text
    return gap_analysis



In [66]:
# Call the function to analyze the resume against the job description using OpenAI
gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                             resume_text, 
                                                             model = "openai")

# Displays the analysis results in Markdown format
print_markdown("#### OpenAI Response:")
print_markdown(gap_analysis_openai)

#### OpenAI Response:

Certainly! Here's a structured analysis comparing the job description with Jessica Brown's resume:

---

### 1. **Key Requirements from Job Description:**

- **Education:** Bachelor's degree in Marketing, Communications, or similar.
- **Experience:** 2+ years of digital marketing experience.
- **Skills:**
  - Familiarity with SEO, SEM, Google Analytics, and social media platforms.
  - Ability to interpret basic marketing data.
  - Good communication and writing skills.
  - Knowledge of CRM systems (e.g., HubSpot) is helpful.
  - Experience with Adobe Creative Suite is beneficial.
- **Responsibilities:**
  - Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
  - Use Google Analytics to measure performance and prepare basic reports.
  - Support social media management tasks including content scheduling and community engagement.
  - Perform keyword research and assist in optimizing content for SEO.
  - Work with designers to coordinate campaign materials.
  - Stay informed about current digital marketing trends.

---

### 2. **Relevant Experience in Resume:**

- **Education:**
  - Bachelor of Commerce, Marketing from Ryerson University (now Toronto Metropolitan University).
- **Experience:**
  - 2 years as a Marketing Assistant at Brewster Coffee Co., assisting with digital marketing campaigns including email and social media.
  - Experience in creating blog posts and social media updates to enhance audience engagement.
  - Managed social media accounts and increased follower numbers.
  - Conducted market research and competitor analysis.
- **Skills:**
  - Digital marketing (SEO basics, Email Marketing).
  - Proficient with social media tools (Hootsuite, Buffer).
  - Basic knowledge of Adobe Photoshop.

---

### 3. **Gaps/Mismatches:**

- **Google Analytics:** The resume does not explicitly mention experience with Google Analytics, which is a key requirement for performance measurement and report preparation.
- **SEM Experience:** While SEO basics are mentioned, SEM experience is not highlighted.
- **CRM Systems:** There is no mention of knowledge or experience with CRM systems like HubSpot.
- **Adobe Creative Suite:** Only basic knowledge of Adobe Photoshop is listed, which may not fully meet the job's preference for Adobe Creative Suite experience.
- **Keyword Research and Content Optimization:** Explicit experience in keyword research and content optimization for SEO is not detailed.

---

### 4. **Potential Strengths:**

- **Content Creation:** Experience in creating blog posts and social media updates could be a strong asset for content-driven campaigns.
- **Social Media Growth:** Proven ability to manage social media accounts and grow follower numbers suggests effective community engagement skills.
- **Market Research and Competitor Analysis:** Skills in market research and competitor analysis could provide valuable insights for campaign strategy and competitive positioning.
- **Multitasking and Event Coordination:** Demonstrated ability to handle multiple tasks and support event coordination may indicate strong organizational and project management skills.

---

Overall, Jessica Brown's resume aligns well with the job description in several areas, such as education and social media management. However, addressing the gaps in Google Analytics, SEM, and CRM experience would strengthen her application for the Digital Marketing Specialist role.

In [69]:
# Call the function to analyze the resume against the job description using Gemini
gap_analysis_gemini = analyze_resume_against_job_description(job_description_text, 
                                                              resume_text, 
                                                              model = "gemini")

# Displays the analysis results in Markdown format
print_markdown("#### Gemini Response:")
print_markdown(gap_analysis_gemini)

#### Gemini Response:

Here's an analysis of Jessica Brown's resume against the Digital Marketing Specialist job description:

---

### 1. Key Requirements from Job Description:

*   **Education:** Bachelor's degree in Marketing, Communications, or similar.
*   **Experience:** 2+ years of digital marketing experience.
*   **Core Digital Marketing Skills:**
    *   Familiarity/experience with SEO (planning, execution, keyword research, content optimization).
    *   Familiarity/experience with SEM (planning, execution).
    *   Familiarity/experience with Social Media (planning, execution, management, content scheduling, community engagement).
    *   Familiarity/experience with Email Marketing (planning, execution).
*   **Analytics & Reporting:** Use Google Analytics to measure performance and prepare basic reports; ability to interpret basic marketing data.
*   **Coordination:** Work with designers to help coordinate campaign materials.
*   **Soft Skills:** Good communication and writing skills; keep informed about current digital marketing trends.
*   **Helpful/Beneficial:** Knowledge of CRM systems (e.g., HubSpot); experience with Adobe Creative Suite.

### 2. Relevant Experience in Resume:

*   **Education:** Bachelor of Commerce, Marketing (direct match).
*   **Experience Duration:** 2 years of experience (Jan 2022 – Present) aligns with the "2+ years" requirement.
*   **Digital Campaigns:** "Assisted with digital marketing campaigns including email and social media."
*   **Email Marketing:** Explicitly mentioned in experience and skills.
*   **Social Media:** Strong alignment with "Assisted with digital marketing campaigns including... social media," "Created social media updates to improve audience engagement," "Managed social media accounts and grew follower numbers," and "Social Media Tools (Hootsuite, Buffer)."
*   **SEO:** "SEO basics" listed in skills. "Created blog posts" suggests content creation relevant to SEO.
*   **Writing Skills:** Demonstrated through "Created blog posts and social media updates."
*   **Coordination:** "Supported coordination of marketing events" shows general coordination ability.
*   **Adobe:** "Basic knowledge of Adobe Photoshop" aligns with the "beneficial" Adobe Creative Suite experience.

### 3. Gaps/Mismatches:

*   **SEM Experience:** No explicit mention or demonstration of experience with Search Engine Marketing (SEM). This is a significant gap as it's a core responsibility.
*   **Google Analytics & Data Interpretation:** No mention of using Google Analytics for performance measurement, reporting, or interpreting basic marketing data. This is a key responsibility outlined in the job description.
*   **Specific SEO Actions:** While "SEO basics" is listed, there's no specific experience detailed for "performing keyword research" or "optimizing content for SEO" beyond general blog post creation.
*   **Coordination with Designers:** While general coordination is mentioned, there's no specific experience coordinating with *designers* for *campaign materials*.
*   **CRM Systems:** No mention of familiarity or experience with CRM systems like HubSpot.
*   **Digital Marketing Trends:** No explicit mention of keeping informed about current digital marketing trends.

### 4. Potential Strengths:

*   **Content Creation:** "Created blog posts and social media updates" highlights practical content creation skills, which are highly valuable in digital marketing.
*   **Audience Engagement & Growth:** "Improved audience engagement" and "grew follower numbers" are quantifiable achievements that demonstrate direct impact and effectiveness in social media management.
*   **Market Research & Competitor Analysis:** "Conducted market research and competitor analysis" shows a proactive and analytical mindset, which can contribute to strategic campaign planning.
*   **Social Media Tool Proficiency:** Explicit mention of Hootsuite and Buffer indicates hands-on experience with industry-standard social media management platforms.
*   **Adaptability & Support:** The summary's mention of "comfortable handling multiple tasks and providing general marketing support" suggests a flexible and team-oriented individual, which is beneficial in an agency setting.

In [80]:
class ResumeOutput(BaseModel):
    updated_resume: str
    diff_markdown: str

def generate_resume(
    job_description_text: str, resume_text: str, gap_analysis_openai: str, model: str = "openai") -> dict:

    """
    Generate a tailored resume using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        resume_text (str): The candidate's resume text.
        gap_analysis_openai (str): The gap analysis result from OpenAI.
        model (str): The model to use for resume generation.

    Returns:
        dict: A dictionary containing the updated resume and diff markdown.
    """
    # Construct the prompt for the AI model to generate the tailored resume.
    # The prompt includes context, instructions, and input data (original resume,
    # target job description, and gap analysis).
    prompt = (
        """
    ### Context:
    You are an expert resume writer and editor. Your goal is to rewrite the original resume to match the target job description, using the provided tailoring suggestions and analysis.

    ---

    ### Instruction:
    1. Rewrite the entire resume to best match the **Target Job Description** and **Gap Analysis to the Job Description**.
    2. Improve clarity, add job-relevant keywords, and quantify achievements.
    3. Specifically address the gaps identified in the analysis by:
       - Adding missing skills and technologies mentioned in the job description
       - Reframing experience to highlight relevant accomplishments
       - Strengthening sections that were identified as weak in the analysis
    4. Prioritize addressing the most critical gaps first
    5. Incorporate industry-specific terminology from the job description
    6. Ensure all quantifiable achievements are properly highlighted with metrics
    7. Return two versions of the resume:
        - `updated_resume`: The final rewritten resume (as plain text)
        - `diff_markdown`: A version of the resume with inline highlights using color:
            - Additions or rewritten content should be **green**:  
            `<span style="color:green">your added or changed text</span>`
            - Removed content should be **red and struck through**:  
            `<span style="color:red;text-decoration:line-through">removed text</span>`
            - Leave unchanged lines unmarked.
        - Keep all section headers and formatting consistent with the original resume.

    ---

    ### Output Format:

    ```json
    {
    "updated_resume": "<full rewritten resume as plain text>",
    "diff_markdown": "<HTML-colored version of the resume highlighting additions and deletions>"
    }
    ```
    ---
    ### Input:

    **Original Resume:**

    """
        + resume_text
        + """


    **Target Job Description:**

    """
        + job_description_text
        + """


    **Analysis of Resume vs. Job Description:**

    """
        + gap_analysis_openai
    )

    if model == "openai":
        updated_resume_json = openai_resume_rewrite(prompt, temperature=0.7, response_format=ResumeOutput)

    elif model == "gemini":
        updated_resume_json = gemini_resume_rewrite(prompt, temperature=0.5, response_format=ResumeOutput)
    else:
        raise ValueError(f"Invalid model: {model}")

    return updated_resume_json

In [81]:
# Call the generate_resume function with the provided job description, resume text, and gap analysis.
updated_resume_json = generate_resume(job_description_text, resume_text, gap_analysis_openai, model="openai")
# Display the updated resume in Markdown format.
print_markdown(updated_resume_json.updated_resume)

**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Dynamic and results-oriented digital marketing professional with over 2 years of experience in executing comprehensive digital campaigns, social media management, and content creation. Proven ability to leverage Google Analytics for performance measurement and optimize content for SEO. Adept at managing multiple projects and staying updated with the latest digital marketing trends.

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Collaborated in planning and executing digital marketing campaigns, including SEO, SEM, social media, and email, resulting in a 20% increase in engagement.
- Utilized Google Analytics to measure campaign performance and prepared comprehensive performance reports.
- Performed keyword research and optimized content for SEO, contributing to a 15% increase in organic traffic.
- Managed and scheduled content on social media platforms, increasing followers by 30%.
- Worked closely with designers to coordinate and produce campaign materials.
- Conducted thorough market research and competitor analysis to inform marketing strategies.

**Skills**
- Digital Marketing: SEO, SEM, Email Marketing
- Analytics: Google Analytics, Data Interpretation
- Social Media Tools: Hootsuite, Buffer
- Design & Creative: Adobe Photoshop, Adobe Creative Suite
- CRM Systems: Basic knowledge of HubSpot
- Microsoft Office Suite, Google Workspace

**Education**  
**Bachelor of Commerce, Marketing** | Toronto Metropolitan University, Toronto, ON | May 2021

In [82]:
print_markdown(updated_resume_json.diff_markdown)

**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
<span style="color:red;text-decoration:line-through">Marketing professional with 2 years of experience assisting in digital campaigns, content creation, and social media activities.</span><span style="color:green">Dynamic and results-oriented digital marketing professional with over 2 years of experience in executing comprehensive digital campaigns, social media management, and content creation.</span> Proven ability to leverage Google Analytics for performance measurement and optimize content for SEO. <span style="color:green">Adept at managing multiple projects and staying updated with the latest digital marketing trends.</span>

**Experience**

**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- <span style="color:red;text-decoration:line-through">Assisted with digital marketing campaigns including email and social media.</span><span style="color:green">Collaborated in planning and executing digital marketing campaigns, including SEO, SEM, social media, and email, resulting in a 20% increase in engagement.</span>
- <span style="color:green">Utilized Google Analytics to measure campaign performance and prepared comprehensive performance reports.</span>
- <span style="color:red;text-decoration:line-through">Created blog posts and social media updates to improve audience engagement.</span>
- <span style="color:green">Performed keyword research and optimized content for SEO, contributing to a 15% increase in organic traffic.</span>
- Managed <span style="color:green">and scheduled content on</span> social media platforms, <span style="color:green">increasing followers by 30%.</span>
- <span style="color:red;text-decoration:line-through">Supported coordination of marketing events.</span>
- <span style="color:green">Worked closely with designers to coordinate and produce campaign materials.</span>
- Conducted <span style="color:green">thorough</span> market research and competitor analysis <span style="color:green">to inform marketing strategies.</span>

**Skills**
- Digital Marketing: <span style="color:red;text-decoration:line-through">SEO basics</span><span style="color:green">SEO, SEM,</span> Email Marketing
- <span style="color:green">Analytics: Google Analytics, Data Interpretation</span>
- Social Media Tools: Hootsuite, Buffer
- <span style="color:red;text-decoration:line-through">Microsoft Office Suite, Google Workspace</span>
- <span style="color:green">Design & Creative: Adobe Photoshop, Adobe Creative Suite</span>
- <span style="color:green">CRM Systems: Basic knowledge of HubSpot</span>
- Microsoft Office Suite, Google Workspace

**Education**  
**Bachelor of Commerce, Marketing** | <span style="color:red;text-decoration:line-through">Ryerson University (now</span> Toronto Metropolitan University<span style="color:red;text-decoration:line-through">)</span>, Toronto, ON | May 2021

In [83]:
class CoverLetterOutput(BaseModel):
    cover_letter: str

def generate_cover_letter(job_description_text: str, updated_resume: str, model: str = "openai") -> dict:
    """
    Generate a cover letter using OpenAI or Gemini.

    Args:
        job_description_text (str): The job description text.
        updated_resume (str): The candidate's updated resume text.
        model (str): The model to use for cover letter generation.

    Returns:
        dict: A dictionary containing the cover letter.
    """

    # Construct the prompt for the AI model, including context and instructions for writing the cover letter.
    prompt = (
        """
    ### Context:
    You are a professional career coach and expert cover letter writer.

    ---

    ### Instruction:
    Write a compelling, personalized cover letter based on the **Updated Resume** and the **Target Job Description**. The letter should:
    1. Be addressed generically (e.g., "Dear Hiring Manager")
    2. Be no longer than 4 paragraphs
    3. Highlight key achievements and experiences from the updated resume
    4. Align with the responsibilities and qualifications in the job description
    5. Reflect the applicant's enthusiasm and fit for the role
    6. End with a confident and polite closing statement

    ---

    ### Output Format (JSON):
    ```json
    {
    "cover_letter": "<final cover letter text>"
    }
    ```
    ---

    ### Input:

    **Updated Resume:**

    """
        + updated_resume
        + """
    **Target Job Description:**

    """
        + job_description_text
    )

    if model == "openai":
        updated_cover_letter = openai_resume_rewrite(prompt, temperature=0.7, response_format=CoverLetterOutput)
    elif model == "gemini":
        updated_cover_letter = gemini_resume_rewrite(prompt, temperature=0.5, response_format=CoverLetterOutput)
    else:
        raise ValueError(f"Invalid model: {model}")

    return updated_cover_letter

In [84]:
updated_cover_letter = generate_cover_letter(job_description_text, updated_resume_json.updated_resume, model="openai")

print_markdown(updated_cover_letter.cover_letter)

Dear Hiring Manager,

I am writing to express my interest in the Digital Marketing Specialist position at BrightWave Digital Agency as advertised. With a robust foundation in digital marketing and a proven track record of executing successful campaigns, I am excited about the opportunity to contribute to your team and support BrightWave's dynamic marketing initiatives.

In my current role as a Marketing Assistant at Brewster Coffee Co., I have effectively collaborated in planning and executing diverse digital marketing campaigns, resulting in a notable 20% increase in engagement. My hands-on experience with Google Analytics has allowed me to measure campaign performance accurately and prepare insightful reports, aligning closely with the responsibilities outlined in your job description. Additionally, my efforts in optimizing content for SEO contributed to a 15% rise in organic traffic, highlighting my capability to enhance digital marketing strategies.

My experience in managing and scheduling content on social media platforms has been instrumental in boosting followers by 30%. I am proficient in leveraging tools such as Hootsuite and Buffer to streamline content scheduling and community engagement, ensuring seamless digital communication. Collaborating with designers to coordinate campaign materials has honed my teamwork skills, which I am eager to bring to BrightWave's creative projects.

I am enthusiastic about the prospect of joining BrightWave Digital Agency, where I can continue to grow within a forward-thinking environment. My academic background in marketing and hands-on experience equip me to not only meet but exceed the expectations of the Digital Marketing Specialist role. I am confident in my ability to contribute to your team's success and am eager to bring my passion for digital marketing to BrightWave.

Thank you for considering my application. I am looking forward to the opportunity to discuss how my skills and experiences align with the goals of BrightWave Digital Agency.

Sincerely,

Jessica Brown

In [86]:
def run_resume_rocket(resume_text: str, job_description_text: str) -> tuple[str, str]:
    """
    Run the resume rocket workflow.

    Args:
        resume_text (str): The candidate's resume text.
        job_description_text (str): The job description text.

    Returns:
        tuple: A tuple containing the updated resume and cover letter.
    """

    gap_analysis_openai = analyze_resume_against_job_description(job_description_text, 
                                                             resume_text, 
                                                             model = "openai")

    # Display the gap analysis results in Markdown format for better readability.
    print_markdown(gap_analysis_openai)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    updated_resume_json = generate_resume(job_description_text, resume_text, gap_analysis_openai, model="openai")

    print_markdown(updated_resume_json.diff_markdown)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    print_markdown(updated_resume_json.updated_resume)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    updated_cover_letter = generate_cover_letter(job_description_text, updated_resume_json.updated_resume, model="openai")

    print_markdown(updated_cover_letter.cover_letter)

    # Print separators for clarity in the output.
    print("\n--------------------------------")
    print("--------------------------------\n")

    return updated_resume_json.updated_resume, updated_cover_letter.cover_letter



In [87]:
resume, cover_letter = run_resume_rocket(resume_text, job_description_text)

Certainly! Below is the structured analysis comparing the provided Job Description and Resume.

---

### 1. Key Requirements from Job Description:
- **Education:** Bachelor's degree in Marketing, Communications, or similar.
- **Experience:** 2+ years in digital marketing.
- **Skills:**
  - Familiarity with SEO, SEM, Google Analytics, and social media.
  - Ability to interpret basic marketing data.
  - Good communication and writing skills.
  - Knowledge of CRM systems (e.g., HubSpot) helpful.
  - Experience with Adobe Creative Suite is beneficial.
- **Responsibilities:**
  - Assist in planning and executing digital marketing campaigns (SEO, SEM, social media, email).
  - Use Google Analytics to measure performance and prepare reports.
  - Support social media management tasks including content scheduling and community engagement.
  - Perform keyword research and assist in optimizing content for SEO.
  - Coordinate with designers for campaign materials.
  - Stay informed about current digital marketing trends.

### 2. Relevant Experience in Resume:
- **Education:** Bachelor of Commerce, Marketing.
- **Experience:** 2 years of experience as a Marketing Assistant, assisting in digital campaigns including email and social media.
- **Skills:**
  - Digital Marketing knowledge including SEO basics and email marketing.
  - Experience with social media tools like Hootsuite and Buffer.
  - Basic knowledge of Adobe Photoshop.
  - Market research and competitor analysis skills.
- **Responsibilities:**
  - Assisted with digital marketing campaigns.
  - Created blog posts and social media updates.
  - Managed social media accounts and increased follower numbers.

### 3. Gaps/Mismatches:
- **Google Analytics:** No mention of experience or familiarity with Google Analytics.
- **SEM:** No mention of experience or familiarity with SEM in the resume.
- **CRM Systems:** No mention of experience with CRM systems like HubSpot.
- **Adobe Creative Suite:** Only basic knowledge of Adobe Photoshop is mentioned; broader experience with Adobe Creative Suite would be beneficial.
- **Keyword Research and SEO Optimization:** While SEO basics are mentioned, specific experience in keyword research and content optimization is not clearly stated.
- **Campaign Coordination with Designers:** No specific mention of coordinating with designers for campaign materials.

### 4. Potential Strengths:
- **Content Creation:** Experience in creating blog posts and social media updates, which could enhance content marketing strategies.
- **Social Media Management:** Proven ability to manage and grow social media accounts, which is crucial for community engagement tasks.
- **Market Research:** Skills in market research and competitor analysis could provide valuable insights for campaign strategies.
- **Event Coordination:** Experience in supporting marketing events, which although not explicitly mentioned in the job description, could be beneficial for comprehensive campaign management.

--- 

### Recommendations:
To enhance the resume's alignment with the job description, Jessica could:
- Gain or highlight experience with Google Analytics and SEM.
- Consider obtaining or demonstrating familiarity with CRM systems like HubSpot.
- Expand on experiences with Adobe Creative Suite beyond Photoshop.
- Provide more detail on any experiences related to keyword research and SEO content optimization.
- Add examples of working with designers if applicable, to demonstrate coordination skills.


--------------------------------
--------------------------------



<span style="color:green">**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown</span>

**Summary**  
<span style="color:red;text-decoration:line-through">Marketing professional with 2 years of experience assisting in digital campaigns, content creation, and social media activities. Comfortable handling multiple tasks and providing general marketing support.</span>  
<span style="color:green">Dynamic digital marketing specialist with over 2 years of experience in planning, executing, and optimizing digital campaigns. Proficient in SEO, SEM, social media, and email marketing, with a proven track record of leveraging analytics to enhance campaign performance. Adept at coordinating with cross-functional teams to deliver compelling marketing materials.</span>

**Experience**

<span style="color:red;text-decoration:line-through">**Marketing Assistant | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**</span>  
<span style="color:green">**Digital Marketing Specialist | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**</span>  
<span style="color:red;text-decoration:line-through">- Assisted with digital marketing campaigns including email and social media.</span>  
<span style="color:green">- Spearheaded digital marketing campaigns encompassing SEO, SEM, and email marketing, leading to a 30% increase in online engagement.</span>  
<span style="color:red;text-decoration:line-through">- Created blog posts and social media updates to improve audience engagement.</span>  
<span style="color:green">- Utilized Google Analytics to track campaign performance, preparing detailed reports for management.</span>  
<span style="color:green">- Conducted comprehensive keyword research, optimizing content to rank higher in search engine results, increasing organic traffic by 25%.</span>  
<span style="color:red;text-decoration:line-through">- Managed social media accounts and grew follower numbers.</span>  
<span style="color:green">- Managed and expanded social media presence, achieving a 40% growth in follower count through strategic content scheduling and community engagement.</span>  
<span style="color:red;text-decoration:line-through">- Supported coordination of marketing events.</span>  
<span style="color:green">- Collaborated with design teams to produce and coordinate marketing collateral, ensuring alignment with brand messaging.</span>  
<span style="color:red;text-decoration:line-through">- Conducted market research and competitor analysis.</span>  
<span style="color:green">- Continually researched and implemented emerging digital marketing trends to maintain competitive edge.</span>

**Skills**  
<span style="color:red;text-decoration:line-through">- Digital Marketing (SEO basics, Email Marketing)</span>  
<span style="color:green">- Digital Marketing Strategies (SEO, SEM, Email Marketing)</span>  
<span style="color:green">- Google Analytics, CRM systems (HubSpot)</span>  
<span style="color:red;text-decoration:line-through">- Social Media Tools (Hootsuite, Buffer)</span>  
<span style="color:green">- Social Media Management (Hootsuite, Buffer)</span>  
<span style="color:red;text-decoration:line-through">- Microsoft Office Suite, Google Workspace</span>  
<span style="color:green">- Adobe Creative Suite (Photoshop, Illustrator)</span>  
<span style="color:green">- Market Research and Data Analysis</span>

**Education**  
<span style="color:red;text-decoration:line-through">**Bachelor of Commerce, Marketing** | Ryerson University (now Toronto Metropolitan University), Toronto, ON | May 2021</span>  
<span style="color:green">**Bachelor of Commerce, Marketing** | Toronto Metropolitan University, Toronto, ON | May 2021</span>


--------------------------------
--------------------------------



**Jessica Brown**  
jessica.brown@email.com | (416) 555-7890 | linkedin.com/in/jessicabrown

**Summary**  
Dynamic digital marketing specialist with over 2 years of experience in planning, executing, and optimizing digital campaigns. Proficient in SEO, SEM, social media, and email marketing, with a proven track record of leveraging analytics to enhance campaign performance. Adept at coordinating with cross-functional teams to deliver compelling marketing materials.

**Experience**

**Digital Marketing Specialist | Brewster Coffee Co. | Toronto, ON | Jan 2022 – Present**
- Spearheaded digital marketing campaigns encompassing SEO, SEM, and email marketing, leading to a 30% increase in online engagement.
- Utilized Google Analytics to track campaign performance, preparing detailed reports for management.
- Conducted comprehensive keyword research, optimizing content to rank higher in search engine results, increasing organic traffic by 25%.
- Managed and expanded social media presence, achieving a 40% growth in follower count through strategic content scheduling and community engagement.
- Collaborated with design teams to produce and coordinate marketing collateral, ensuring alignment with brand messaging.
- Continually researched and implemented emerging digital marketing trends to maintain competitive edge.

**Skills**
- Digital Marketing Strategies (SEO, SEM, Email Marketing)
- Google Analytics, CRM systems (HubSpot)
- Social Media Management (Hootsuite, Buffer)
- Adobe Creative Suite (Photoshop, Illustrator)
- Market Research and Data Analysis

**Education**  
**Bachelor of Commerce, Marketing** | Toronto Metropolitan University, Toronto, ON | May 2021


--------------------------------
--------------------------------



Dear Hiring Manager,

I am writing to express my interest in the Digital Marketing Specialist position at BrightWave Digital Agency, as advertised. With a Bachelor of Commerce in Marketing and over two years of hands-on experience in digital marketing, I am eager to bring my skills and enthusiasm to your dynamic team.

At Brewster Coffee Co., I spearheaded digital marketing campaigns that included SEO, SEM, and email strategies, resulting in a 30% increase in online engagement. My proficiency in Google Analytics has been instrumental in tracking and reporting campaign performance, enabling data-driven decisions that boosted organic traffic by 25%. Additionally, my experience managing and expanding social media presence aligns seamlessly with BrightWave's requirement for content scheduling and community engagement.

I am particularly excited about the opportunity to work collaboratively with BrightWave's designers to coordinate compelling campaign materials. My background in utilizing Adobe Creative Suite to ensure brand messaging alignment will allow me to effectively contribute to your creative processes. Furthermore, my commitment to staying abreast of emerging digital marketing trends will help maintain BrightWave's competitive edge.

I am confident that my strategic approach, coupled with my passion for digital marketing, makes me an ideal fit for this role. I am eager to bring my expertise in digital marketing strategies and data analysis to your esteemed agency. Thank you for considering my application. I look forward to the opportunity to discuss how I can contribute to the continued success of BrightWave Digital Agency.

Sincerely,

Jessica Brown


--------------------------------
--------------------------------

